# 01 — Données
Collecte sous licence libre, tri, légendes, préparation 512×512, split train/test.

In [ ]:
# Setup — works on Colab / Kaggle AND locally (VS Code, Jupyter)
import os, sys, subprocess
from pathlib import Path

ON_CLOUD = "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if ON_CLOUD:
    if not Path("lora-zellige").exists():
        subprocess.run(["git", "clone", "https://github.com/braziyass/lora-zellige.git"], check=True)
    os.chdir("lora-zellige")
else:
    # Locally the notebook starts in notebooks/ : go up to the repo root
    root = Path.cwd().resolve()
    while not (root / "requirements.txt").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)
sys.path.insert(0, os.getcwd())
PY = f'"{sys.executable}"'   # used as !{PY} -m ... in the cells below
print("Working directory:", os.getcwd())

print("Python used by this notebook:", sys.executable)

# Packages: installed automatically on Colab/Kaggle only.
# Locally, install once in the Terminal (see README), or set INSTALL = True.
INSTALL = ON_CLOUD
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for notebooks 01-02 data steps)")
except ImportError:
    print("torch not installed: only needed from notebook 02 on (use Colab/Kaggle for training)")

In [ ]:
# Optionnel : persister données et LoRA sur Google Drive entre les sessions
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/lora-zellige/{raw,outputs}
# !ln -sfn /content/drive/MyDrive/lora-zellige/raw data/raw
# !ln -sfn /content/drive/MyDrive/lora-zellige/outputs outputs

## 1. Téléchargement depuis Wikimedia Commons (licences compatibles uniquement)

In [ ]:
!{PY} data/download_commons.py --category "Zellige" --limit 80

## 2. Tri et légendes (MANUEL)
Téléchargez `data/sources.csv`, remplissez `keep` (1/0) et `caption`, puis rechargez-le. Aperçu des candidates :

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
src = pd.read_csv("data/sources.csv")
print(src["license"].value_counts())
fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for ax, f in zip(axes.flat, src["file"][:24]):
    ax.imshow(Image.open(f"data/raw/{f}")); ax.set_title(f[:20], fontsize=7); ax.axis("off")
plt.tight_layout()

## 3. Préparation

In [ ]:
!{PY} -m src.prepare_data --config configs/base.yaml
import json; print(json.dumps(json.load(open('data/processed/report.json')), indent=1)[:1500])

In [ ]:
from src.utils import read_jsonl
meta = read_jsonl("data/processed/train/metadata.jsonl")
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, m in zip(axes.flat, meta):
    ax.imshow(Image.open(f"data/processed/train/{m['file_name']}")); ax.axis("off")
    ax.set_title(m["text"][:45], fontsize=6)
plt.tight_layout(); plt.savefig("results/figures/corpus.png", dpi=150)